## Expectations and Densities

Let's consider the example of the Bing design update that was worth a 12% revenue increase: making long ad titles instead of two lines of text.

<img src="./src/bing.jpeg" width="1300px">


 **Let's assume this extra revenue came from an increased click-through rate from $p_A=0.1$ to $p_B=0.105$.** We'll consider a **sample of 200 people** seeing each design, and compare the ECDFs. But first, there are two questions we could ask:
1. What actually happened when people were given either design? (i.e. looking at a single sample of people for design A and for design B, what are the differences?)
2. How much would the proportion who clicked change if I collected the data again? How confident can I be in the proportions I got? 

The first question is answered by an ECDF on the random variable itself. The second is answered by an ECDF of the sampling distribution, which we can get by simulating multiple samples, computing a proportion, and looking at the ECDF of that proportion.

||ECDF of one sample|ECDF of sampling distribution|
|---|------|-------|
|**What goes into it**|n observations|B replicate values of a statistic (i.e. B simulations of size n)|
|**Domain**|Whatever the variable's own units are|The statistic's units (often proportion from 0 to 1 or rescaled mean)|
|**Shape**|Whatever shape the real variable actually has|Tends toward bell-shaped as n grows (CLT, which we will come back to later!)|
|**What happens as n grows**|Nothing about its width changes, because more data doesn't change the underlying shape (it's just resolved more finely)|It gets narrower, more data makes the statistic less variable|






### Helper functions and imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(20260717)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [ ]:
def ecdf_xy(samples):
    x = np.sort(np.asarray(samples))
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y


def plot_ecdf_vs_cdf(samples, cdf, grid=None, title="", xlabel="", discrete=False, logx=False):
    x, y = ecdf_xy(samples)
    fig, ax = plt.subplots()
    ax.step(x, y, where="post", lw=2, label="simulation ECDF")

    if grid is None:
        lo, hi = np.quantile(x, [0.001, 0.999])
        grid = np.linspace(lo, hi, 400)
    if discrete:
        ax.step(grid, cdf(grid), where="post", lw=2, label="scipy CDF")
    else:
        ax.plot(grid, cdf(grid), lw=2, label="scipy CDF")

    if logx:
        ax.set_xscale("log")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("cumulative probability")
    ax.legend()
    return ax


def show_quantiles(samples, dist, probs=(0.1, 0.25, 0.5, 0.75, 0.9)):
    return pd.DataFrame({
        "p": probs,
        "simulation": np.quantile(samples, probs),
        "scipy": dist.ppf(probs),
    })

## Use RNG to get a 200-person sample for each UX design

In [ ]:

# Using RNG, build a sample with p = 0.1 and a second sample with p = 0.105, each with N = 200. 
N = 200
pA = 0.1
pB = 0.105

#### 1 bernoulli trial, with p = p_A, repeated N times, and then summed to get the cumulative successes. Do the same for p_B.
sampleA = rng.binomial(1, pA, size= N)
sampleB = rng.binomial(1, pB, size= N)

trial_pathsA = sampleA.cumsum() #summing along the trial axis to get cumulative successes
trial_pathsB = sampleB.cumsum()
# Make a plot of the running trials and compare.
fig, ax = plt.subplots()
ax.plot(np.arange(1, N + 1), trial_pathsA.T, alpha=0.45, c="tab:blue", label=f"p = {pA}")
ax.plot(np.arange(1, N + 1), trial_pathsB.T, alpha=0.45, c="tab:orange", label=f"p = {pB}")
ax.plot(np.arange(1, N + 1), pA * np.arange(1, N + 1), color="blue", lw=2, label="expected successes ")
ax.plot(np.arange(1, N + 1), pB * np.arange(1, N + 1), color="orange", lw=2, label="expected successes")


ax.set_title("Binomial counts are accumulated yes/no events")
ax.set_xlabel("trial")
ax.set_ylabel("cumulative successes")
ax.legend()
plt.show()



## ECDF of 200-person sample
Our sample is made of of 200 0s and 1s, with some proportion of each (around .1 or .105 1's). So the ECDF, asking "how many are below or equal to x" will sweep between 0 and 1 and will jump up immediately to the sample proportion of 0s.

In [ ]:
#ECDF of sample A vs sample B
fig, ax = plt.subplots(ncols=2, figsize=(10, 4))

#left plot: sample A vs sample B
x, y = ecdf_xy(sampleA)  
ax[0].step(x, y, where='post', color='blue', lw=2, label=f"p = {pA}, sample proportion = {sampleA.mean():.3f}")
x, y = ecdf_xy(sampleB)   
ax[0].step(x, y, where='post', color='orange', lw=2, label=f"p = {pB}, sample proportion = {sampleB.mean():.3f}")

ax[0].legend()

#right plot: expected CDFs for sample A and sample B
grid = np.linspace(0, 1, 400)
cdfA = stats.binom.cdf(grid, n=1, p=pA)
cdfB = stats.binom.cdf(grid, n=1, p=pB)
ax[1].plot(grid, cdfA, color='blue', lw=2, label=f"p = {pA}")
ax[1].plot(grid, cdfB, color='orange', lw=2, label=f"p = {pB}")
ax[1].set_title("ECDF vs CDF")
ax[1].set_xlabel("successes")
ax[1].set_ylabel("cumulative probability")
ax[1].set_xlim(-0.1, 1.1)
ax[1].set_ylim(-0.1, 1.1)
ax[1].legend()
plt.show()





It's not very informative, because if we are asking "how many people are $\leq 0$", this is ~ $1-p$ (so at x = 0, we jump up to ~ 1-p), and then at x = 1, we jump up to 1.

## What if we repeated that 200 person sample 50 times?

In [ ]:

# Using RNG, build a sample with p = 0.1 and a second sample with p = 0.105, each with N = 200. 
N = 200
pA = 0.1
pB = 0.105

#### 1 bernoulli trial, with p = p_A, repeated N times per sample, for 50 samples, and then summed to get cumulative successes per sample. 
# Do the same for p_B.
trial_pathsA = rng.binomial(1, pA, size=(50, N)).cumsum(axis=1) #summing along the trial axis to get cumulative successes
trial_pathsB = rng.binomial(1, pB, size=(50, N)).cumsum(axis=1)



# Make the plot of running trials and compare.
fig, ax = plt.subplots()
ax.plot(np.arange(1, N + 1), trial_pathsA.T, alpha=0.45, c="tab:blue")
ax.plot(np.arange(1, N + 1), trial_pathsB.T, alpha=0.45, c="tab:orange")
ax.plot(np.arange(1, N + 1), pA * np.arange(1, N + 1), color="blue", lw=2, label="expected successes ")
ax.plot(np.arange(1, N + 1), pB * np.arange(1, N + 1), color="orange", lw=2, label="expected successes")
ax.set_title("Binomial counts are accumulated yes/no events")
ax.set_xlabel("trial")
ax.set_ylabel("cumulative successes")
ax.legend()
plt.show()

## ECDF of Sampling distribution
Instead of looking at how the clicks accumulate within each, lets take build the ECDF of the sample proportion over many samples (i.e. what does the sampling distribution look like for proportion?). We will get 50 samples of 200 people, calculate the number of people who clicked, normalize it by N, and look at the ECDF. 

Now, instead of our ECDF on 0s and 1s, our ECDF is on the total clicks (*a statistic*, the sum of 1s) in each sample. So our x-axis here is going to be the normalized counts, and we'll ask "what proportion of our samples had a normalized count $\leq$ x?"

Now let's look at the ECDF for each of our designs. How big of a sample do we need to start to really see a difference? (We'll come back to these ideas mathematicall later with A/B testing; for now, we just get to think in pictures)

In [ ]:
N = 200 # each sample has 200 people who clicked or didn't click
B = 50 # 50 samples

pA = 0.1
pB = 0.105
fig, ax = plt.subplots(figsize=(5, 4))

#### N bernoulli trials, with p = p_A, repeated for B samples. 
# rng.binomial will return the number of successes in N trials for
# each of the B samples. 
counts = rng.binomial(N, pA, size=B)
p_hat_A = counts / N
x, y = ecdf_xy(p_hat_A)
ax.step(x, y, where='post', color='blue', lw=2, label=f"p = {pA}")

counts = rng.binomial(N, pB, size=B)
p_hat_B = counts / N
x, y = ecdf_xy(p_hat_B)
ax.step(x, y, where='post', color='orange', lw=2, label=f"p = {pB}")
ax.legend()
plt.show()
    

As we grow N, you may start to recognize the shape of the ECDF: it approaches the Normal CDF. (This is a small taste of the central limit theorem, which we will come back to. For now, we are going to focus on how ECDFs $\rightarrow$ CDFs of known distributions.) If we standardize the click rate, it looks like the Normal CDF.

In [ ]:

N = 2000 # each sample has 200 people who clicked or didn't click
B = 500 # 50 samples

pA = 0.1
pB = 0.105

#### N bernoulli trials, with p = p_A, repeated for B samples. 
# rng.binomial will return the number of successes in N trials for
# each of the B samples. 
counts = rng.binomial(N, pA, size=B)
z_binom = (counts - N * pA) / np.sqrt(N * pA * (1 - pA))
normal = stats.norm(0, 1)

plot_ecdf_vs_cdf(z_binom, normal.cdf, grid=np.linspace(-4, 4, 500), title="CLT from binomial counts", xlabel="standardized count")


## The Expected Sample Proportion of the ECDF 
- What is the expected value of the ECDF?
$$
\begin{alignat*}{2}
\mathbb{E}[\hat{F}(x)] &=& \mathbb{E}_X \left[ \frac{1}{n} \sum_{i=1}^n \mathbb{I} \{ X_i \le x \} \right] \quad \left( \text{ Definition } \right) \\
&=&  \frac{1}{n} \sum_{i=1}^n \mathbb{E}_X \left[\mathbb{I} \{ X_i \le x \} \right] \quad \left( \text{Linearity} \right) \\
&=&   \frac{1}{n} \sum_{i=1}^n F(x) \quad \left( \text{Expectation of indicator is probability}\right) \\
&=&  F(x) \quad \left( \text{Sum of $n$ identical terms over $n$} \right)
\end{alignat*}
$$
- The ECDF $\hat{F}(x)$ is an unbiased estimator of the true CDF $F(x)$

## CDF TO PMF

- Let's put $X$ on a grid $\{x_1, x_2, ..., x_J\}$, where the space between $x_j$ and $x_{j+1}$ is $h$
- Let's "split the difference" between grid points, and compute the probability that $X$ is between $x_j - h/2$ and $x_j +h/2$ equal to $F(x_j+h/2) - F(x_j-h/2)$
- Let's approximate the value of $X$ itself on the interval $[x_j-h/2,x_j+h/2)$ by $x_j$
- This **discretizes** the random variable $X$, giving it a probability mass function and a finite number of values to take


## Approximating the Expectation
- On the grid, our approximation of the expected value is
$$
\mathbb{E}[X] \approx \sum_{j=1}^J \underbrace{x_j}_{\approx x} \times \underbrace{F(x_j+h/2)-F(x_j-h/2)}_{\approx p[x_j-h/2 \le X < x_j+h]}
$$
- We want to let $h$ get small and make this "look like an integral"
- Multiply and divide by $h$, and we get this:
$$
\mathbb{E}[X] \approx \underbrace{\sum_{j=1}^J}_{\rightarrow \int} \quad \underbrace{x_j}_{\rightarrow X} \times \underbrace{\frac{F(x_j+h/2)-F(x_j-h/2)}{h}}_{\rightarrow F'(x)} \times \underbrace{h}_{\rightarrow {dx}}
$$


<img src="./src/pdf_from_cdf.png" width="900px">

## PMF to Probability Density Function

- So for a continuous random variable, 
$$
\mathbb{E}[X] = \int_x x F'(x) dx
$$
- The term $F'(x)$ is the **probability density function** of $F(x)$, and we write $f(x) = F'(x)$
- Some random variables are more easily characterized by their distribution function $F(x)$ (e.g. uniform), and some are more easily characterized by their density $f(x)=F'(x)$ (e.g. normal)


## Probability Density Functions
- Some distributions are easier to describe in terms of their density
- Instead of starting with the cdf $F(x)$ and taking the derivative to get the pdf $f(x)=F'(x)$, it makes more sense sometimes to start with the density $f(x)$ and then integrate, so 
$$
F(x) = \int_{-\infty}^x f(z)dz
$$
- Since $F(x) = \text{pr}[ X \le x]$, the complementary probability is $1-F(x) = \text{pr}[X > x]$
- Likewise, $\text{pr}[a \le X < b ] = \int_{a}^{b} f(x)dx = F(b) - F(a)$


## Expectation and Variance
- We've defined the expectation and variance a few times, depending on the scenario, but the principles are the same:
    1. To get the expected value, weight each $x$ by the "probability" it occurs, now $f(x)$, and sum
    2. To get the variance, weight each squared deviation of $X$ from its expected value, $(X-\mathbb{E}[X])^2$, by the "probability" it occurs, now $f(x)$, and sum
- With **continuous random variables** with a density function $f(x)$, we define the expectation as
$$
\mathbb{E}[X] = \int_{x} x f(x) dx
$$
and the variance as
$$
\mathbb{V}[X] = \int_x (x - \mathbb{E}[X])^2 f(x) dx
$$


## Exercise
- Suppose a variable is uniformly distributed, so it has distribution function:
$$
F(x) = \begin{cases}
0, & x < 0 \\
x, & 0 \le x \le 1 \\
1, & x > 1
\end{cases}
$$
- What is the probability density function, the expectation, and variance?

## Distribution Helper
- The next examples use small interactive plots to connect parameters, densities, distribution functions, expectation, and variance
- Move the sliders and watch how the PDF/PMF, CDF, $\mathbb{E}[X]$, and $\mathbb{V}[X]$ change


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import gamma
import ipywidgets as widgets


def _format_stat(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "undefined"
    if np.isposinf(value):
        return "infinite"
    if np.isneginf(value):
        return "-infinite"
    return f"{value:.4g}"


def _finite_grid_from_dist(dist, support=None, q_low=0.001, q_high=0.999, n=600):
    if support is not None:
        lo, hi = support
    else:
        lo = dist.ppf(q_low)
        hi = dist.ppf(q_high)
        if not np.isfinite(lo):
            lo = dist.ppf(0.01)
        if not np.isfinite(hi):
            hi = dist.ppf(0.99)
    if lo == hi:
        lo -= 1
        hi += 1
    pad = 0.04 * (hi - lo)
    return np.linspace(lo - pad, hi + pad, n)


def plot_continuous_distribution(name, dist, mean, variance, support=None):
    x = _finite_grid_from_dist(dist, support=support)
    pdf = dist.pdf(x)
    cdf = dist.cdf(x)

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].plot(x, pdf, color="#2d6cdf", lw=2)
    axes[0].set_title("PDF")
    axes[0].set_xlabel("x")
    axes[0].set_ylabel("f(x)")
    axes[0].grid(alpha=0.25)

    axes[1].plot(x, cdf, color="#00876c", lw=2)
    axes[1].set_title("CDF")
    axes[1].set_xlabel("x")
    axes[1].set_ylabel("F(x)")
    axes[1].set_ylim(-0.04, 1.04)
    axes[1].grid(alpha=0.25)

    if mean is not None and np.isfinite(mean):
        for ax in axes:
            ax.axvline(mean, color="#9f2f2f", ls="--", lw=1.5, label="E[X]")
        axes[0].legend(loc="best")

    # if mean is not None and np.isfinite(mean) and np.isfinite(variance) and variance >= 0:
    #     sd = np.sqrt(variance)
    #     axes[0].axvspan(mean - sd, mean + sd, color="#9f2f2f", alpha=0.10, label="+/- 1 SD")

    fig.suptitle(
        f"{name}: E[X] = {_format_stat(mean)}    V[X] = {_format_stat(variance)}",
        y=1.05,
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()


def plot_discrete_distribution(name, values, probabilities, mean, variance):
    values = np.asarray(values)
    probabilities = np.asarray(probabilities)
    cdf = np.cumsum(probabilities)

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].bar(values, probabilities, color="#2d6cdf", width=0.8)
    axes[0].set_title("PMF")
    axes[0].set_xlabel("x")
    axes[0].set_ylabel("p[X=x]")
    axes[0].grid(axis="y", alpha=0.25)

    axes[1].step(values, cdf, where="post", color="#00876c", lw=2)
    axes[1].scatter(values, cdf, color="#00876c", s=20)
    axes[1].set_title("CDF")
    axes[1].set_xlabel("x")
    axes[1].set_ylabel("F(x)")
    axes[1].set_ylim(-0.04, 1.04)
    axes[1].grid(alpha=0.25)

    if mean is not None and np.isfinite(mean):
        for ax in axes:
            ax.axvline(mean, color="#9f2f2f", ls="--", lw=1.5, label="E[X]")
        axes[0].legend(loc="best")

    fig.suptitle(
        f"{name}: E[X] = {_format_stat(mean)}    V[X] = {_format_stat(variance)}",
        y=1.05,
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()


## Example: Bernoulli Distribution

- A Bernoulli random variable records one success-or-failure trial
- Its PMF is
$$
p[X=x] = p^x(1-p)^{1-x}, \qquad x\in\{0,1\}
$$
- Its CDF is
$$
F(x)=\begin{cases}
0, & x<0 \\
1-p, & 0\le x<1 \\
1, & x\ge 1
\end{cases}
$$


In [ ]:
@widgets.interact(
    p=widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01, description="p"),
)
def bernoulli_demo(p):
    values = np.array([0, 1])
    probabilities = np.array([1 - p, p])
    mean = p
    variance = p * (1 - p)
    plot_discrete_distribution("Bernoulli(p)", values, probabilities, mean, variance)


## Example: Binomial Distribution

- A binomial random variable counts the number of successes in $n$ independent Bernoulli trials with success probability $p$
- Its PMF is
$$
p[X=k]=\binom{n}{k}p^k(1-p)^{n-k}, \qquad k=0,1,\ldots,n
$$
- Its CDF is the cumulative sum of the PMF:
$$
F(k)=\sum_{j=0}^{\lfloor k \rfloor}\binom{n}{j}p^j(1-p)^{n-j}
$$


In [ ]:
@widgets.interact(
    n=widgets.IntSlider(value=10, min=1, max=60, step=1, description="n"),
    p=widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01, description="p"),
)
def binomial_demo(n, p):
    values = np.arange(n + 1)
    dist = stats.binom(n=n, p=p)
    probabilities = dist.pmf(values)
    mean = n * p
    variance = n * p * (1 - p)
    plot_discrete_distribution("Binomial(n, p)", values, probabilities, mean, variance)


## Example: Uniform
- The uniform distribution spreads probability evenly over an interval $[a,b]$
- Its CDF is
$$
F(x) = \begin{cases}
0, & x < a \\
\dfrac{x-a}{b-a}, & a \le x \le b \\
1, & x > b
\end{cases}
$$
- Its PDF is
$$
f(x) = \begin{cases}
\dfrac{1}{b-a}, & a \le x \le b \\
0, & \text{otherwise}
\end{cases}
$$

In [ ]:
@widgets.interact(
    a=widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.25, description="a"),
    b=widgets.FloatSlider(value=1.0, min=-4.75, max=10.0, step=0.25, description="b"),
)
def uniform_demo(a, b):
    if b <= a:
        print("Choose b > a.")
        return
    dist = stats.uniform(loc=a, scale=b - a)
    mean = (a + b) / 2
    variance = (b - a) ** 2 / 12
    plot_continuous_distribution("Uniform(a, b)", dist, mean, variance, support=(a, b))


## Example: The Exponential Distribution
- The exponential distribution is supported on $[0,\infty)$
- Its CDF is
$$
F(t) = \begin{cases}
0, & t<0 \\
1 - e^{-\lambda t}, & t \ge 0
\end{cases}
$$
- Its PDF is
$$
f(t) = \begin{cases}
\lambda e^{-\lambda t}, & t \ge 0 \\
0, & t<0
\end{cases}
$$


In [ ]:
@widgets.interact(
    rate=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="lambda"),
)
def exponential_demo(rate):
    dist = stats.expon(scale=1 / rate)
    mean = 1 / rate
    variance = 1 / rate**2
    plot_continuous_distribution("Exponential(lambda)", dist, mean, variance, support=(0, dist.ppf(0.995)))


## Example: The Logistic Distribution
- The logistic distribution has CDF
$$
F(x) = \dfrac{1}{1+e^{-(x-\mu)/\sigma}}
$$
- Its PDF is
$$
f(x) = \dfrac{e^{-(x-\mu)/\sigma}}{\sigma\left(1+e^{-(x-\mu)/\sigma}\right)^2}
$$


In [ ]:
@widgets.interact(
    mu=widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.25, description="mu"),
    sigma=widgets.FloatSlider(value=1.0, min=0.2, max=4.0, step=0.1, description="sigma"),
)
def logistic_demo(mu, sigma):
    dist = stats.logistic(loc=mu, scale=sigma)
    mean = mu
    variance = (np.pi**2 * sigma**2) / 3
    plot_continuous_distribution("Logistic(mu, sigma)", dist, mean, variance)


## Example: Normal Distribution

- The normal density is
$$
f(x; \mu, \sigma) = \frac{1}{\sqrt{2\pi} \sigma}e^{-(x-\mu)^2/(2\sigma^2)}
$$
- The normal CDF has no elementary closed form, so we usually compute it numerically
- Notice that it is bell-shaped and symmetric around $\mu$


In [ ]:
@widgets.interact(
    mu=widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.25, description="mu"),
    sigma=widgets.FloatSlider(value=1.0, min=0.2, max=4.0, step=0.1, description="sigma"),
)
def normal_demo(mu, sigma):
    dist = stats.norm(loc=mu, scale=sigma)
    mean = mu
    variance = sigma**2
    plot_continuous_distribution("Normal(mu, sigma)", dist, mean, variance)


## Example: Beta Distribution

- The beta distribution is supported on $[0,1]$, so it is useful for probabilities, proportions, and rates
- Its PDF is
$$
f(x;\alpha,\beta)=\frac{x^{\alpha-1}(1-x)^{\beta-1}}{B(\alpha,\beta)}, \qquad 0<x<1
$$
- Its CDF does not have a simple closed form for general $\alpha$ and $\beta$


In [ ]:
@widgets.interact(
    alpha=widgets.FloatSlider(value=2.0, min=0.2, max=10.0, step=0.2, description="alpha"),
    beta=widgets.FloatSlider(value=2.0, min=0.2, max=10.0, step=0.2, description="beta"),
)
def beta_demo(alpha, beta):
    dist = stats.beta(a=alpha, b=beta)
    mean = alpha / (alpha + beta)
    variance = alpha * beta / ((alpha + beta) ** 2 * (alpha + beta + 1))
    plot_continuous_distribution("Beta(alpha, beta)", dist, mean, variance, support=(0, 1))


## Example: Geometric Distribution

- A geometric random variable records the waiting time until the first success
- Here $T=1$ means the first trial succeeds
- Its PMF is
$$
p[T=t]=(1-p)^{t-1}p, \qquad t=1,2,3,\ldots
$$
- Its CDF is
$$
F(t)=p[T\le t]=1-(1-p)^{\lfloor t \rfloor}, \qquad t\ge 1
$$


In [ ]:
@widgets.interact(
    p=widgets.FloatSlider(value=0.3, min=0.02, max=1.0, step=0.01, description="p"),
)
def geometric_demo(p):
    dist = stats.geom(p=p)
    upper = int(max(8, dist.ppf(0.995)))
    values = np.arange(1, upper + 1)
    probabilities = dist.pmf(values)
    mean = 1 / p
    variance = (1 - p) / p**2
    plot_discrete_distribution("Geometric(p)", values, probabilities, mean, variance)


## Example: Poisson Distribution for Count Data

- A Poisson random variable counts how many events occur in a fixed interval when events arrive at average rate $\lambda$
- Its PMF is
$$
p[X=k]=e^{-\lambda}\frac{\lambda^k}{k!}, \qquad k=0,1,2,\ldots
$$
- Its CDF is the cumulative sum of the PMF:
$$
F(k)=\sum_{j=0}^{\lfloor k \rfloor}e^{-\lambda}\frac{\lambda^j}{j!}
$$


In [ ]:
@widgets.interact(
    lam=widgets.FloatSlider(value=3.0, min=0.1, max=20.0, step=0.1, description="lambda"),
)
def poisson_demo(lam):
    dist = stats.poisson(mu=lam)
    upper = int(max(8, dist.ppf(0.999)))
    values = np.arange(0, upper + 1)
    probabilities = dist.pmf(values)
    mean = lam
    variance = lam
    plot_discrete_distribution("Poisson(lambda)", values, probabilities, mean, variance)


## Example: Lognormal Distribution

- A lognormal random variable is positive and right-skewed
- If $\log X \sim N(\mu,\sigma^2)$, then $X$ is lognormally distributed
- Its PDF is
$$
f(x;\mu,\sigma)=\frac{1}{x\sigma\sqrt{2\pi}}e^{-(\log x-\mu)^2/(2\sigma^2)}, \qquad x>0
$$
- Its CDF can be written using the standard normal CDF $\Phi$:
$$
F(x)=\Phi\left(\frac{\log x-\mu}{\sigma}\right), \qquad x>0
$$


In [ ]:
@widgets.interact(
    mu=widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1, description="mu"),
    sigma=widgets.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1, description="sigma"),
)
def lognormal_demo(mu, sigma):
    dist = stats.lognorm(s=sigma, scale=np.exp(mu))
    mean = np.exp(mu + sigma**2 / 2)
    variance = (np.exp(sigma**2) - 1) * np.exp(2 * mu + sigma**2)
    plot_continuous_distribution("Lognormal(mu, sigma)", dist, mean, variance, support=(0, dist.ppf(0.995)))


## Example: Pareto Distribution

- The Pareto distribution is a simple model of heavy-tailed positive quantities
- With minimum value $x_m$ and tail parameter $\alpha$, its CDF is
$$
F(x)=\begin{cases}
0, & x<x_m \\
1-\left(\dfrac{x_m}{x}\right)^\alpha, & x\ge x_m
\end{cases}
$$
- Its PDF is
$$
f(x)=\begin{cases}
\dfrac{\alpha x_m^\alpha}{x^{\alpha+1}}, & x\ge x_m \\
0, & x<x_m
\end{cases}
$$
- As $\alpha$ gets smaller, rare large values become much more important


In [ ]:
@widgets.interact(
    alpha=widgets.FloatSlider(value=3.0, min=1.1, max=8.0, step=0.1, description="alpha"),
    xm=widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.25, description="x_m"),
)
def pareto_demo(alpha, xm):
    dist = stats.pareto(b=alpha, scale=xm)
    mean = alpha * xm / (alpha - 1) if alpha > 1 else np.inf
    variance = alpha * xm**2 / ((alpha - 1) ** 2 * (alpha - 2)) if alpha > 2 else np.inf
    plot_continuous_distribution("Pareto(alpha, x_m)", dist, mean, variance, support=(xm, dist.ppf(0.99)))


## Example: Mixture of Normals

- A mixture distribution combines two or more probability models
- For a two-component normal mixture with weight $w$, its PDF is
$$
f(x)=w f_1(x)+(1-w)f_2(x)
$$
where $f_1$ and $f_2$ are normal densities
- Its CDF is the same weighted average of the component CDFs:
$$
F(x)=w F_1(x)+(1-w)F_2(x)
$$
- This is a useful way to see that distributions can be multi-peaked even when each component is bell-shaped


In [ ]:
@widgets.interact(
    w=widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01, description="w"),
    mu1=widgets.FloatSlider(value=-2.0, min=-6.0, max=3.0, step=0.25, description="mu1"),
    mu2=widgets.FloatSlider(value=2.0, min=-3.0, max=6.0, step=0.25, description="mu2"),
    sigma1=widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1, description="sigma1"),
    sigma2=widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1, description="sigma2"),
)
def normal_mixture_demo(w, mu1, mu2, sigma1, sigma2):
    lo = min(mu1 - 4 * sigma1, mu2 - 4 * sigma2)
    hi = max(mu1 + 4 * sigma1, mu2 + 4 * sigma2)
    x = np.linspace(lo, hi, 700)
    pdf = w * stats.norm.pdf(x, mu1, sigma1) + (1 - w) * stats.norm.pdf(x, mu2, sigma2)
    cdf = w * stats.norm.cdf(x, mu1, sigma1) + (1 - w) * stats.norm.cdf(x, mu2, sigma2)
    mean = w * mu1 + (1 - w) * mu2
    second_moment = w * (sigma1**2 + mu1**2) + (1 - w) * (sigma2**2 + mu2**2)
    variance = second_moment - mean**2

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].plot(x, pdf, color="#2d6cdf", lw=2)
    axes[0].axvline(mean, color="#9f2f2f", ls="--", lw=1.5, label="E[X]")
    axes[0].set_title("PDF")
    axes[0].set_xlabel("x")
    axes[0].set_ylabel("f(x)")
    axes[0].grid(alpha=0.25)
    axes[0].legend(loc="best")

    axes[1].plot(x, cdf, color="#00876c", lw=2)
    axes[1].axvline(mean, color="#9f2f2f", ls="--", lw=1.5)
    axes[1].set_title("CDF")
    axes[1].set_xlabel("x")
    axes[1].set_ylabel("F(x)")
    axes[1].set_ylim(-0.04, 1.04)
    axes[1].grid(alpha=0.25)

    fig.suptitle(
        f"Normal mixture: E[X] = {_format_stat(mean)}    V[X] = {_format_stat(variance)}",
        y=1.05,
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()
